In [ ]:
# Import package
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc
import igraph
import scvelo as scv
import loompy as lmp
import anndata as ad
import harmonypy as hm
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

import scvi

# scRNA, snRNA 불러오기

In [ ]:
adata_scraw=sc.read("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/scRNA_v1/scRNA_raw.h5ad")

In [ ]:
adata_scrna=sc.read("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/scRNA_v1/scRNA_adata_01.h5ad")

In [ ]:
adata_scrna

In [ ]:
adata_scraw.obs['11cluster']=adata_scrna.obs['11cluster']
adata_scraw.obs['celltypist_liver']=adata_scrna.obs['celltypist_liver']

In [ ]:
sc.pl.umap(adata_scrna, color = 'cellannot_detail_JHB')

In [ ]:
sc.pl.umap(adata_scrna, color = '11cluster')

In [ ]:
sc.pl.umap(adata_scrna, color = 'celltypist_liver')

In [ ]:
adata_scmyl = adata_scraw[adata_scraw.obs['11cluster'].isin(['1'])].copy()

In [ ]:
adata_scmyl.write("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/scRNA_v1/scRNA_adata_myeloid_raw.h5ad")

In [ ]:
adata_snraw=sc.read("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/snRNA_v1/snRNA_raw.h5ad")
adata_snrna=sc.read("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/snRNA_v1/snRNA_adata_01.h5ad")

In [ ]:
adata_snrna

In [ ]:
sc.pl.umap(adata_snrna, color = 'celltypist_liver')

In [ ]:
sc.tl.leiden(adata_snrna, resolution=0.05, key_added="7cluster")
sc.pl.umap(adata_snrna, color=['7cluster'])

In [ ]:
sn_adata_mp = sn_adata_pre[sn_adata_pre.obs['cellannot'].isin(['MP'])].copy()
sn_adata_endo = sn_adata_pre[sn_adata_pre.obs['cellannot'].isin(['Endothelia'])].copy()
sn_adata_mes = sn_adata_pre[sn_adata_pre.obs['cellannot'].isin(['Mesenchyme'])].copy()

# sc_sn_Endo analysis

In [ ]:
sc_adata_endo

In [ ]:
print(sc_adata_endo.raw)

In [ ]:
sc_adata_endo = sc_adata_endo.raw.to_adata()

In [ ]:
sc_adata_endo

In [ ]:
sn_adata_endo.obs['disease'] = sn_adata_endo.obs['batch']

In [ ]:
sn_adata_endo.obs['disease']

In [ ]:
cluster2annotation = {
    "normal": "Hepatocytes",
    "MASLD": "Macrophage"
}
sn_adata_endo.obs['disease'] = sn_adata_endo.obs['disease'].map(cluster2annotation).astype("category")

In [ ]:
sc_adata_endo.obs['disease'].value_counts()

In [ ]:
# sc_adata_endo analysis

In [ ]:
masld_list = ['healthy','NAFLD']
sc_adata_endo = sc_adata_endo[sc_adata_endo.obs['disease'].isin(masld_list)].copy()

In [ ]:
sc_adata_endo.obs['disease'].value_counts()

In [ ]:
sc_adata_endo

In [ ]:
sc.pp.normalize_total(sc_adata_endo, target_sum=1e4)
sc.pp.log1p(sc_adata_endo)

In [ ]:
# 변수 유전자 설정 
sc.pp.highly_variable_genes(sc_adata_endo, flavor='seurat', n_top_genes=2000)
# Scale data
sc.pp.scale(sc_adata_endo, max_value=10)
# Run PCA
sc.tl.pca(sc_adata_endo, svd_solver='arpack')
print(sc_adata_endo.obsm['X_pca'][:5, :5])
sc.pl.pca(sc_adata_endo, color='sample')

import matplotlib.pyplot as plt
var_ratio = sc_adata_endo.uns['pca']['variance_ratio']
plt.plot(range(1, len(var_ratio) + 1), np.cumsum(var_ratio), marker='o')
plt.xlabel('Number of Principal Components')
plt.ylabel('Cumulative Explained Variance')
plt.title('PCA Cumulative Variance')
plt.grid(True)
plt.show()

In [ ]:
# t-sne 
sc.tl.tsne(sc_adata_endo, use_rep='X_pca', n_pcs=30)
# umap
sc.pp.neighbors(sc_adata_endo, n_pcs=30, use_rep='X_pca')
sc.tl.umap(sc_adata_endo)
sc.pl.umap(sc_adata_endo, color=['sample'])

In [ ]:
sc.pl.umap(sc_adata_endo, color=['cellannot_detail_MP_Endo_Mes'])

In [ ]:
# Harmony로 batch effect 제거
harmony_out = hm.run_harmony(sc_adata_endo.obsm['X_pca'],sc_adata_endo.obs, 'sample')

# Harmony 결과를 저장
sc_adata_endo.obsm['X_pca_harmony'] = harmony_out.Z_corr.T

# UMAP을 사용한 시각화
sc.pp.neighbors(sc_adata_endo, use_rep='X_pca_harmony')  # Harmony 결과를 사용
sc.tl.umap(sc_adata_endo)
sc.pl.umap(sc_adata_endo, color=['cellannot_detail_MP_Endo_Mes'])

In [ ]:
sc.pl.umap(sc_adata_endo, color=['disease'])

In [ ]:
sn_adata_endo

In [ ]:
sc.pp.normalize_total(sn_adata_endo, target_sum=1e4)
sc.pp.log1p(sn_adata_endo)

In [ ]:
# 변수 유전자 설정 
sc.pp.highly_variable_genes(sn_adata_endo, flavor='seurat', n_top_genes=2000)
# Scale data
sc.pp.scale(sn_adata_endo, max_value=10)
# Run PCA
sc.tl.pca(sn_adata_endo, svd_solver='arpack')
print(sn_adata_endo.obsm['X_pca'][:5, :5])
sc.pl.pca(sn_adata_endo, color='sample')

import matplotlib.pyplot as plt
var_ratio = sn_adata_endo.uns['pca']['variance_ratio']
plt.plot(range(1, len(var_ratio) + 1), np.cumsum(var_ratio), marker='o')
plt.xlabel('Number of Principal Components')
plt.ylabel('Cumulative Explained Variance')
plt.title('PCA Cumulative Variance')
plt.grid(True)
plt.show()

In [ ]:
# t-sne 
sc.tl.tsne(sn_adata_endo, use_rep='X_pca', n_pcs=30)
# umap
sc.pp.neighbors(sn_adata_endo, n_pcs=30, use_rep='X_pca')
sc.tl.umap(sn_adata_endo)
sc.pl.umap(sn_adata_endo, color=['sample'])

In [ ]:
sc.pl.umap(sn_adata_endo, color=['disease'])

In [ ]:
# Harmony로 batch effect 제거
harmony_out = hm.run_harmony(sn_adata_endo.obsm['X_pca'],sn_adata_endo.obs, 'sample')

# Harmony 결과를 저장
sn_adata_endo.obsm['X_pca_harmony'] = harmony_out.Z_corr.T

# UMAP을 사용한 시각화
sc.pp.neighbors(sn_adata_endo, use_rep='X_pca_harmony')  # Harmony 결과를 사용
sc.tl.umap(sn_adata_endo)
sc.pl.umap(sn_adata_endo, color=['disease'])

In [ ]:
sc.tl.leiden(sn_adata_endo, resolution=0.4, key_added="cellannot")
sc.pl.umap(sn_adata_endo, color=['cellannot'])

In [ ]:
sc_adata_endo.obs['endo_cluster'] = sc_adata_endo.obs['cellannot_detail_MP_Endo_Mes']

In [ ]:
sn_adata_endo.obs['endo_cluster'] = sn_adata_endo.obs['cellannot']

In [ ]:
common_genes = list(set((sc_adata_endo.var_names).intersection(sn_adata_endo.var_names)))
sc = sc_adata_endo[:, common_genes]
sn = sn_adata_endo[:, common_genes]

In [ ]:
# cluster 평균 발현 peason correlation 계산

In [ ]:
sc_means = sc.to_df().groupby(sc.obs['endo_cluster']).mean()
sn_means = sn.to_df().groupby(sn.obs['endo_cluster']).mean()

In [ ]:
sc_means

In [ ]:
cor_matrix = pd.DataFrame(
    np.corrcoef(sc_means.values, sn_means.values)[:len(sc_means), len(sc_means):],
    index=sc_means.index,
    columns=sn_means.index
)

In [ ]:
matching = cor_matrix.idxmax(axis=1)
matching_scores = cor_matrix.max(axis=1)

In [ ]:
print("SC-SN 1:1 Matching 결과:")
for sc_clust in matching.index:
    print(f"SC_Cluster {sc_clust} ↔ SN_Cluster {matching[sc_clust]} (corr={matching_scores[sc_clust]:.2f})")


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8,6))
sns.heatmap(cor_matrix, annot=True, fmt=".2f", cmap="bwr")
plt.title("SC vs SN Cluster Correlation Heatmap")
plt.show()

In [ ]:
# 각 cluster 별 marker gene expression 비교

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
from scipy.stats import hypergeom
from itertools import product

In [ ]:
print(sc_adata_endo,
      sn_adata_endo)

In [ ]:
def get_top_markers(adata, groupby, n_genes=100):
    sc.tl.rank_genes_groups(adata, groupby=groupby, method='wilcoxon')
    markers = {}
    for clust in adata.obs[groupby].unique():
        top_genes = adata.uns['rank_genes_groups']['names'][clust][:n_genes]
        markers[clust] = set(top_genes)
    return markers

sc_markers = get_top_markers(sc_adata_endo, groupby="endo_cluster", n_genes=100)
sn_markers = get_top_markers(sn_adata_endo, groupby="endo_cluster", n_genes=100)

In [ ]:
def compare_marker_sets(sc_markers, sn_markers, total_genes):
    results = []
    for sc_clust, sn_clust in product(sc_markers.keys(), sn_markers.keys()):
        overlap = len(sc_markers[sc_clust] & sn_markers[sn_clust])
        M = total_genes                # 전체 유전자 수
        n = len(sn_markers[sn_clust])  # sn 클러스터 마커 유전자 수
        N = len(sc_markers[sc_clust])  # sc 클러스터 마커 유전자 수
        pval = hypergeom.sf(overlap-1, M, n, N)  # Hypergeometric test
        jaccard = overlap / len(sc_markers[sc_clust] | sn_markers[sn_clust])
        results.append([sc_clust, sn_clust, overlap, pval, jaccard])
    return pd.DataFrame(results, columns=["SC_cluster","SN_cluster","overlap","pval","jaccard"])


In [ ]:
total_genes = len(set(sc_adata.var_names) & set(sn_adata.var_names))
match_df = compare_marker_sets(sc_markers, sn_markers, total_genes)

In [ ]:
best_matches = (
    match_df.sort_values(["pval","jaccard"], ascending=[True,False])
    .groupby("SC_cluster").first().reset_index()
)
print(best_matches)

In [ ]:
best_matches = (
    match_df.sort_values(["pval","jaccard"], ascending=[True,False])
    .groupby("SC_cluster").first().reset_index()
)
print(best_matches)

In [ ]:
# Fisher exact test

In [ ]:
import numpy as np
import pandas as pd
from itertools import product
from scipy.stats import fisher_exact

In [ ]:
def fisher_marker_matching(sc_markers, sn_markers, total_genes):
    results = []
    total_tests = len(sc_markers) * len(sn_markers)

    for sc_clust, sn_clust in product(sc_markers.keys(), sn_markers.keys()):
        overlap = len(sc_markers[sc_clust] & sn_markers[sn_clust])
        sc_only = len(sc_markers[sc_clust]) - overlap
        sn_only = len(sn_markers[sn_clust]) - overlap
        neither = total_genes - (overlap + sc_only + sn_only)

        table = np.array([[overlap, sc_only],
                          [sn_only, neither]])
        odds, p = fisher_exact(table, alternative='greater')

        p_adj = min(p * total_tests, 1.0)  # Bonferroni correction
        score = -np.log10(p_adj + 1e-300)  # log scale (avoid log(0))
        results.append([sc_clust, sn_clust, overlap, p, p_adj, score])

    return pd.DataFrame(results, columns=["SC_cluster","SN_cluster","overlap","pval","p_adj","score"])

total_genes = len(set(sc_adata.var_names) & set(sn_adata.var_names))
match_df = fisher_marker_matching(sc_markers, sn_markers, total_genes)


In [ ]:
best_matches = (
    match_df.sort_values(["p_adj","score"], ascending=[True,False])
    .groupby("SC_cluster").first().reset_index()
)
print(best_matches)

In [ ]:
best_matches = (
    match_df.sort_values(["p_adj","score"], ascending=[True,False])
    .groupby("SC_cluster").first().reset_index()
)
print(best_matches)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

heatmap_data = match_df.pivot(index="SC_cluster", columns="SN_cluster", values="score")
plt.figure(figsize=(8,6))
sns.heatmap(heatmap_data, annot=True, fmt=".1f", cmap="viridis")
plt.title("-log10 Bonferroni-corrected p-value (Fisher Test)")
plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

heatmap_data = match_df.pivot(index="SC_cluster", columns="SN_cluster", values="score")
plt.figure(figsize=(8,6))
sns.heatmap(heatmap_data, annot=True, fmt=".1f", cmap="viridis")
plt.title("-log10 Bonferroni-corrected p-value (Fisher Test)")
plt.show()

In [ ]:
# cell level 로 확인하기

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import fisher_exact

In [ ]:
total_genes = len(set(sc_adata_endo.var_names) & set(sn_adata_endo.var_names))

In [ ]:
def cell_level_matching(sn_adata, sc_markers, total_genes, expr_threshold=0.1):
    results = []
    sn_df = sn_adata.to_df()  # snRNA expression matrix (cells x genes)

    for cell in sn_df.index:
        expressed_genes = set(sn_df.columns[sn_df.loc[cell] > expr_threshold])
        
        best_match, best_p = None, 1.0
        for sc_clust, sc_genes in sc_markers.items():
            overlap = len(expressed_genes & sc_genes)
            sc_only = len(sc_genes) - overlap
            sn_only = len(expressed_genes) - overlap
            neither = total_genes - (overlap + sc_only + sn_only)

            table = np.array([[overlap, sc_only],
                              [sn_only, neither]])
            _, p = fisher_exact(table, alternative='greater')

            if p < best_p:
                best_match, best_p = sc_clust, p

        results.append([cell, best_match, best_p, -np.log10(best_p+1e-300)])

    return pd.DataFrame(results, columns=["SN_cell","Matched_SC_cluster","pval","score"])

In [ ]:

cell_match_df = cell_level_matching(sn_adata, sc_markers, total_genes)
print(cell_match_df.head())

In [ ]:
significant_cells = cell_match_df[cell_match_df["pval"] < 0.05]

In [ ]:
sn_adata_endo.obs['matched_sc_cluster'] = cell_match_df.set_index("SN_cell").loc[sn_adata_endo.obs_names, "Matched_SC_cluster"]
sc.pl.umap(sn_adata_endo, color='matched_sc_cluster')

In [ ]:
sn_adata_endo.obs['matched_sc_cluster'].value_counts()

# sc_sn_MASLD Macrophage analysis

In [ ]:
sc_adata_mp

In [ ]:
sn_adata_mp

In [ ]:
sc_adata_mp = sc_adata_mp.raw.to_adata()

In [ ]:
masld_list = ['healthy','NAFLD']
sc_adata_mp = sc_adata_mp[sc_adata_mp.obs['disease'].isin(masld_list)].copy()

In [ ]:
sc_adata_mp.obs['disease'].value_counts()

In [ ]:
sc.pp.normalize_total(sc_adata_mp, target_sum=1e4)
sc.pp.log1p(sc_adata_mp)

# 변수 유전자 설정 
sc.pp.highly_variable_genes(sc_adata_mp, flavor='seurat', n_top_genes=2000)
# Scale data
sc.pp.scale(sc_adata_mp, max_value=10)
# Run PCA
sc.tl.pca(sc_adata_mp, svd_solver='arpack')
print(sc_adata_mp.obsm['X_pca'][:5, :5])
sc.pl.pca(sc_adata_mp, color='sample')

In [ ]:
import matplotlib.pyplot as plt
var_ratio = sc_adata_mp.uns['pca']['variance_ratio']
plt.plot(range(1, len(var_ratio) + 1), np.cumsum(var_ratio), marker='o')
plt.xlabel('Number of Principal Components')
plt.ylabel('Cumulative Explained Variance')
plt.title('PCA Cumulative Variance')
plt.grid(True)
plt.show()


In [ ]:
# t-sne 
sc.tl.tsne(sc_adata_mp, use_rep='X_pca', n_pcs=30)
# umap
sc.pp.neighbors(sc_adata_mp, n_pcs=30, use_rep='X_pca')
sc.tl.umap(sc_adata_mp)
sc.pl.umap(sc_adata_mp, color=['sample'])

In [ ]:
# Harmony로 batch effect 제거
harmony_out = hm.run_harmony(sc_adata_mp.obsm['X_pca'],sc_adata_mp.obs, 'sample')

# Harmony 결과를 저장
sc_adata_mp.obsm['X_pca_harmony'] = harmony_out.Z_corr.T

# UMAP을 사용한 시각화
sc.pp.neighbors(sc_adata_mp, use_rep='X_pca_harmony')  # Harmony 결과를 사용
sc.tl.umap(sc_adata_mp)
sc.pl.umap(sc_adata_mp, color=['disease'])

In [ ]:
sc_adata_mp

In [ ]:
sc.pl.umap(sc_adata_mp, color=['cellannot_detail_MP_Endo_Mes'])

In [ ]:
sc.pl.tsne(sc_adata_mp, color=['cellannot_detail_MP_Endo_Mes'])